<a href="https://colab.research.google.com/github/rmaomina/llm_service_modu6/blob/main/Colab%20Notebooks/w5_rag_search_improvement/llm_260407_rag_search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import re
import time
import json
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy import stats

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import ContextualCompressionRetriever, EnsembleRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from dotenv import load_dotenv

load_dotenv()

MODEL = "gpt-4o-mini"
llm = ChatOpenAI(model=MODEL)
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small")


In [ ]:
documents = [
    Document(page_content="트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.", metadata={"id": "d1"}),
    Document(page_content="BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.", metadata={"id": "d2"}),
    Document(page_content="GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.", metadata={"id": "d3"}),
    Document(page_content="RAG는 검색 증강 생성 기법으로 외부 지식을 LLM에 결합하여 할루시네이션을 줄입니다.", metadata={"id": "d4"}),
    Document(page_content="벡터 데이터베이스는 임베딩 벡터를 저장하고 유사도 기반 검색을 수행합니다. FAISS, Pinecone 등이 있습니다.", metadata={"id": "d5"}),
    Document(page_content="파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.", metadata={"id": "d6"}),
    Document(page_content="프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.", metadata={"id": "d7"}),
    Document(page_content="토큰화는 텍스트를 모델이 처리할 수 있는 단위로 분할하는 과정입니다. BPE, WordPiece 등이 사용됩니다.", metadata={"id": "d8"}),
]


In [ ]:
vectorstore = FAISS.from_documents(documents, embeddings_model)
bm25_retriever = BM25Retriever.from_documents(documents, k=5)

In [ ]:
doc_embeddings = {}

def get_embedding(text):
    return np.array(embeddings_model.embed_query(text))

for doc in doc_embeddings:
    doc_embeddings[doc.metadata['id']] = get_embedding(doc.page_content)

In [ ]:
# query - doc1, doc2, ...   Cross-encoder
# 사용자 쿼리 : 트랜스포머와 BERT의 관계?
#     1) 트랜스포머는 ~~~~

#     2) BERT는 ~~~ 입니다

In [ ]:
# query -> emb_query
# doc1 -> emb_doc1
# ..
# => 문장들간의 의미적 관련성

In [ ]:
# 검색 -> doc1, doc2, ... doc5 -> 후보문서 -> re-rank ----(Cross-encoder / llm)--> re-ranked -> generate

In [ ]:
# Cross-encoder
# 1. normal vector retriever
# query ---(emb_model)---> query_emb
# doc1 ----(emb_model) ---> doc1_emb

# # 2. cross-encoder retriever
# (query - doc1 ) ---(emb_model)---> (query_doc1_emb)
# (query - doc2 ) ---(emb_model)---> (query_doc2_emb)

# LLM
# query, doc1 --(LLM)---> score, t/f

In [ ]:
def vector_search(query, vectorstore, top_k=5):
    results = vectorstore.similarity_search_with_score(query, k=top_k)
    return [(doc, 1.0 / (1.0 + score)) for doc, score in results]

query = "트랜스포머와 BERT의 관계"
results = vector_search(query, vectorstore)
results

[(Document(id='64de9fb8-aa5e-4f32-9c82-0578d8b41421', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.5248781)),
 (Document(id='367feaa0-430e-44eb-b887-5e3e01cb410f', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.4639076)),
 (Document(id='e11ed846-3859-47a4-bc24-7e5bb3deebcb', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.41887715)),
 (Document(id='bc1e21dc-4e9e-4a8a-831b-4695ab34abca', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  np.float32(0.4127892)),
 (Document(id='5acbe41f-d7c2-46dd-823e-f4f0aae2a28a', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.39725193))]

In [ ]:
def keyword_rerank(query, search_results):
    query_terms = set(query.lower().split())
    reranked = []

    for doc, orig_score in search_results:
        doc_terms = doc.page_content.lower().split()
        keyword_hits = sum(1 for t in doc_terms if t in query_terms)   # [1, 1]
        new_score = orig_score + 0.1 * keyword_hits
        reranked.append((doc, new_score, orig_score))

    reranked.sort(key = lambda x: x[1], reverse=True)
    return reranked

In [ ]:
reranked = keyword_rerank(query, results)
reranked

[(Document(id='64de9fb8-aa5e-4f32-9c82-0578d8b41421', metadata={'id': 'd1'}, page_content='트랜스포머는 Self-Attention 메커니즘을 사용하여 시퀀스 데이터를 병렬로 처리하는 딥러닝 아키텍처입니다.'),
  np.float32(0.5248781),
  np.float32(0.5248781)),
 (Document(id='367feaa0-430e-44eb-b887-5e3e01cb410f', metadata={'id': 'd2'}, page_content='BERT는 양방향 트랜스포머 인코더로 MLM과 NSP 태스크로 사전학습됩니다.'),
  np.float32(0.4639076),
  np.float32(0.4639076)),
 (Document(id='e11ed846-3859-47a4-bc24-7e5bb3deebcb', metadata={'id': 'd3'}, page_content='GPT는 단방향 트랜스포머 디코더로 다음 토큰 예측 방식으로 학습합니다.'),
  np.float32(0.41887715),
  np.float32(0.41887715)),
 (Document(id='bc1e21dc-4e9e-4a8a-831b-4695ab34abca', metadata={'id': 'd6'}, page_content='파인튜닝은 사전학습된 모델을 특정 도메인 데이터로 추가 학습하는 기법입니다. LoRA, QLoRA가 효율적입니다.'),
  np.float32(0.4127892),
  np.float32(0.4127892)),
 (Document(id='5acbe41f-d7c2-46dd-823e-f4f0aae2a28a', metadata={'id': 'd7'}, page_content='프롬프트 엔지니어링은 LLM에 효과적인 지시를 설계하는 기법입니다. Few-shot, CoT 등이 있습니다.'),
  np.float32(0.39725193),
  np.float32(0.39725193)

In [ ]:
def rank_change(original_results, reranked_results):

    문서별 before 순위, after 순위, change